In [1]:
from pathlib import Path

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "parquet"

In [2]:
import pandas as pd

amazon_books = pd.read_parquet(DATA_DIR / "amazon.parquet")
goodreads = pd.read_parquet(DATA_DIR / "goodreads.parquet")
metabooks = pd.read_parquet(DATA_DIR / "metabooks.parquet")

In [3]:
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer, util


# embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

def similarity_score(src, tgt):
    """Compute semantic + fuzzy similarity between column names."""
    emb_src = model.encode(src, convert_to_tensor=True)
    emb_tgt = model.encode(tgt, convert_to_tensor=True)
    semantic_sim = util.cos_sim(emb_src, emb_tgt).item()
    fuzzy_sim = fuzz.token_sort_ratio(src, tgt) / 100.0
    return 0.6 * semantic_sim + 0.4 * fuzzy_sim


def schema_match_semantic(source_cols, target_cols, threshold=0.6):
    """Find best target column for each source column."""
    mapping = {}
    for src in source_cols:
        best_tgt, best_score = None, 0
        for tgt in target_cols:
            score = similarity_score(src, tgt)
            if score > best_score:
                best_score, best_tgt = score, tgt
        mapping[src] = best_tgt if best_score >= threshold else None
    return mapping

In [4]:
def unify_schemas(schema_dict, target_schema_name="metabooks"):
    """
    Use one dataset (e.g., 'metabooks') as target schema.
    Match all other schemas to it.
    """
    if target_schema_name not in schema_dict:
        raise ValueError(f"Target schema '{target_schema_name}' not found in schema_dict")

    target_cols = schema_dict[target_schema_name]
    mappings = {}

    for name, cols in schema_dict.items():
        if name == target_schema_name:
            # Target schema maps to itself
            mappings[name] = {col: col for col in cols}
        else:
            mappings[name] = schema_match_semantic(cols, target_cols)

    return target_cols, mappings

In [5]:
schemas = {
    "amazon": amazon_books.columns,
    "metabooks": metabooks.columns,
    "goodreads": goodreads.columns
}

target_schema, mappings = unify_schemas(schemas, target_schema_name="goodreads")

print("Target Schema:")
print(target_schema)

print("\nSchema Mappings:")
for name, mapping in mappings.items():
    print(f"\n{name}:")
    for k, v in mapping.items():
        print(f"  {k:20} -> {v}")

Target Schema:
Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'bookformat', 'edition', 'page_count', 'publisher', 'publish_year',
       'price', 'isbn_clean'],
      dtype='object')

Schema Mappings:

amazon:
  id                   -> id
  title                -> title
  book-author          -> author
  year-of-publication  -> publish_year
  publisher            -> publisher
  isbn_clean           -> isbn_clean

metabooks:
  id                   -> id
  title                -> title
  author_name          -> author
  rating               -> rating
  num_rating           -> numratings
  language             -> language
  genres               -> genres
  publisher            -> publisher
  page_count           -> page_count
  price                -> price
  publish_year         -> publish_year
  isbn_clean           -> isbn_clean

goodreads:
  id                   -> id
  title                -> title
  author               -> author
  rating       

In [6]:
def apply_schema_mapping(df, mapping):
    rename_dict = {k: v for k, v in mapping.items() if v is not None}
    return df.rename(columns=rename_dict)

amazon_new = apply_schema_mapping(amazon_books, mappings["amazon"])
metabooks_new = apply_schema_mapping(metabooks, mappings["metabooks"])
goodreads_new = apply_schema_mapping(goodreads, mappings["goodreads"])

In [7]:
import pandas as pd
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer, util


# embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

def similarity_score(src, tgt):
    """Compute semantic + fuzzy similarity between column names."""
    emb_src = model.encode(src, convert_to_tensor=True)
    emb_tgt = model.encode(tgt, convert_to_tensor=True)
    semantic_sim = util.cos_sim(emb_src, emb_tgt).item()
    fuzzy_sim = fuzz.token_sort_ratio(src, tgt) / 100.0
    return 0.6 * semantic_sim + 0.4 * fuzzy_sim


def schema_match_semantic(source_cols, target_cols, threshold=0.6):
    """Find best target column for each source column."""
    mapping = {}
    for src in source_cols:
        best_tgt, best_score = None, 0
        for tgt in target_cols:
            score = similarity_score(src, tgt)
            if score > best_score:
                best_score, best_tgt = score, tgt
        mapping[src] = best_tgt if best_score >= threshold else None
    return mapping

In [8]:
def unify_schemas(schema_dict, dataframes, target_schema_name="goodreads"):
    """
    Use one dataset (e.g., 'goodreads') as target schema.
    Match all other schemas to it, rename, and add missing columns.
    """
    if target_schema_name not in schema_dict:
        raise ValueError(f"Target schema '{target_schema_name}' not found in schema_dict")

    target_cols = schema_dict[target_schema_name]
    mappings = {}
    aligned_dfs = {}

    for name, cols in schema_dict.items():
        df = dataframes[name].copy()

        if name == target_schema_name:
            # Target schema maps to itself
            mappings[name] = {col: col for col in cols}
        else:
            mappings[name] = schema_match_semantic(cols, target_cols)
            rename_dict = {k: v for k, v in mappings[name].items() if v is not None}
            df = df.rename(columns=rename_dict)

        # Add missing columns as None
        for col in target_cols:
            if col not in df.columns:
                df[col] = None

        # Reorder columns to match target schema
        df = df[target_cols]
        aligned_dfs[name] = df

    return target_cols, mappings, aligned_dfs

In [9]:
# Schema dictionary
schemas = {
    "amazon": amazon_books.columns,
    "metabooks": metabooks.columns,
    "goodreads": goodreads.columns
}


dfs = {
    "amazon": amazon_books,
    "metabooks": metabooks,
    "goodreads": goodreads
}

# Align all schemas
target_schema, mappings, aligned_dfs = unify_schemas(schemas, dfs, target_schema_name="goodreads")

print("Target Schema:", target_schema)

print("\nSchema Mappings:")
for name, mapping in mappings.items():
    print(f"\n{name}:")
    for k, v in mapping.items():
        print(f"  {k:20} -> {v}")


print("\nAmazon after alignment:", aligned_dfs["amazon"].columns.tolist())
print("\nMetabooks after alignment:", aligned_dfs["metabooks"].columns.tolist())
print("\nGoodreads after alignment:", aligned_dfs["goodreads"].columns.tolist())

Target Schema: Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'bookformat', 'edition', 'page_count', 'publisher', 'publish_year',
       'price', 'isbn_clean'],
      dtype='object')

Schema Mappings:

amazon:
  id                   -> id
  title                -> title
  book-author          -> author
  year-of-publication  -> publish_year
  publisher            -> publisher
  isbn_clean           -> isbn_clean

metabooks:
  id                   -> id
  title                -> title
  author_name          -> author
  rating               -> rating
  num_rating           -> numratings
  language             -> language
  genres               -> genres
  publisher            -> publisher
  page_count           -> page_count
  price                -> price
  publish_year         -> publish_year
  isbn_clean           -> isbn_clean

goodreads:
  id                   -> id
  title                -> title
  author               -> author
  rating       

In [22]:
aligned_dfs["amazon"].to_parquet(DATA_DIR / "amazon_new.parquet", index=False)
aligned_dfs["metabooks"].to_parquet(DATA_DIR / "metabooks_new.parquet", index=False)
aligned_dfs["goodreads"].to_parquet(DATA_DIR / "goodreads_new.parquet", index=False)
amazon_books = pd.read_parquet(DATA_DIR / "amazon_new.parquet")
metabooks = pd.read_parquet(DATA_DIR / "metabooks_new.parquet")
goodreads = pd.read_parquet(DATA_DIR / "goodreads_new.parquet")
goodreads["isbn_clean"] = (
    goodreads["isbn_clean"]
    .replace({"nan": pd.NA, "NaN": pd.NA, "": pd.NA})
)
goodreads=goodreads.dropna(subset=["isbn_clean"]).reset_index(drop=True)

# --- Define overlaps ---
isbn_m2a = set(metabooks['isbn_clean']) & set(amazon_books['isbn_clean'])
isbn_m2g = set(metabooks['isbn_clean']) & set(goodreads['isbn_clean'])
isbn_m2_any = isbn_m2a | isbn_m2g  # Metabooks overlap with either Amazon or Goodreads

# --- Build initial overlap samples ---
metabooks_sample = metabooks[metabooks['isbn_clean'].isin(isbn_m2_any)]
amazon_sample    = amazon_books[amazon_books['isbn_clean'].isin(isbn_m2a)]
goodreads_sample = goodreads[goodreads['isbn_clean'].isin(isbn_m2g)]

# --- Top up to 35k samples with random non-overlapping rows ---
mb_sample_2 = metabooks[~metabooks['isbn_clean'].isin(isbn_m2_any)] \
    .sample(35_000 - len(metabooks_sample), random_state=42)
metabooks_sample = pd.concat([metabooks_sample, mb_sample_2]).reset_index(drop=True)

az_sample_2 = amazon_books[~amazon_books['isbn_clean'].isin(isbn_m2a)] \
    .sample(35_000 - len(amazon_sample), random_state=42)
amazon_sample = pd.concat([amazon_sample, az_sample_2]).reset_index(drop=True)

gr_sample_2 = goodreads[~goodreads['isbn_clean'].isin(isbn_m2g)] \
    .sample(35_000 - len(goodreads_sample), random_state=42)
goodreads_sample = pd.concat([goodreads_sample, gr_sample_2]).reset_index(drop=True)
print(metabooks_sample.shape, amazon_sample.shape, goodreads_sample.shape)



(35000, 14) (35000, 14) (35000, 14)


In [23]:
metabooks_sample.genres

0        ['Parenting', 'Relationships', 'Family Relatio...
1            ["Children's Books", 'Geography', 'Cultures']
2        ['Literature', 'Fiction', 'Short Stories', 'An...
3                               ['Humor', 'Entertainment']
4        ['Literature', 'Fiction', 'Short Stories', 'An...
                               ...                        
34995     ['Literature', 'Fiction', 'Action', 'Adventure']
34996    ['Business', 'Money', 'Management', 'Leadership']
34997                                   ['Travel', 'Asia']
34998                        ['Medical Books', 'Medicine']
34999                      ['Politics', 'Social Sciences']
Name: genres, Length: 35000, dtype: string

In [24]:
import ast
import pandas as pd

def genres_to_string(value):
    if isinstance(value, list):
        tokens = value
    elif isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            tokens = parsed if isinstance(parsed, list) else [parsed]
        except (ValueError, SyntaxError):
            tokens = [value]
    else:
        tokens = [value]

    flat = []
    for t in tokens:
        flat.extend(t if isinstance(t, list) else [t])

    cleaned = []
    for t in flat:
        if t is None or (isinstance(t, str) and t.strip() == ""):
            continue
        if pd.isna(t):
            continue
        cleaned.append(str(t).strip())

    return ", ".join(cleaned)

for df in (goodreads_sample, metabooks_sample):
    df["genres"] = df["genres"].apply(genres_to_string)


In [27]:
amazon_sample.to_parquet(DATA_DIR / "amazon_sample.parquet", index=False)
metabooks_sample.to_parquet(DATA_DIR / "metabooks_sample.parquet", index=False)
goodreads_sample.to_parquet(DATA_DIR / "goodreads_sample.parquet", index=False)